# AI Billing Agent → Structured Parser → Deterministic Rules Engine (mock)

Pipeline (matches the diagram):

1. **AI Billing Agent (Gemini Pro)**: schema-constrained JSON output, no side-effecting tools — the model only ever *proposes* a billing claim as JSON, it never calls a tool that actually bills anything.
2. **Structured Output Parser**: validated against a Pydantic schema, rejected outright on any parse failure.
3. **Deterministic Catalogue Rules Engine**, four checks that never depend on the model:
   1. the code exists in the given GOÄ/GOZ catalogue version
   2. mutual exclusions between codes on the same claim are respected
   3. the billing factor is within its legal range for that code
   4. the service date and quantity are valid

This notebook is a **mock**: no real Gemini call is made. `mock_ai_billing_agent()` returns canned JSON strings for several scenarios (valid, malformed JSON, unknown code, mutual exclusion violation, factor out of range, bad date/quantity) so the parser and rules engine can be exercised end to end without model access.

Run the cells in order.

In [ ]:
%pip install pydantic

In [ ]:
from datetime import date

from pydantic import BaseModel, Field, ValidationError

## 1. Schema the AI Billing Agent's output must satisfy

In [ ]:
class BillingLineItem(BaseModel):
    code: str
    catalogue_version: str  # "GOAE" or "GOZ"
    factor: float
    quantity: int = Field(gt=0)
    service_date: date


class BillingClaim(BaseModel):
    patient_id: str
    line_items: list[BillingLineItem]

## 2. Mock AI Billing Agent (Gemini Pro stand-in)

Returns raw JSON text only — exactly what a schema-constrained Gemini Pro call would hand back. No tool is invoked here, so nothing has been billed yet.

In [ ]:
TODAY = date(2026, 8, 30)

MOCK_AGENT_RESPONSES = {
    "valid": """
    {"patient_id": "p-100", "line_items": [
        {"code": "1", "catalogue_version": "GOAE", "factor": 2.3, "quantity": 1, "service_date": "2026-08-29"}
    ]}
    """,
    "malformed_json": """
    {"patient_id": "p-101", "line_items": [  // trailing comment breaks JSON
        {"code": "1", "catalogue_version": "GOAE", "factor": 2.3, "quantity": 1, "service_date": "2026-08-29"}
    ]}
    """,
    "unknown_code": """
    {"patient_id": "p-102", "line_items": [
        {"code": "9999", "catalogue_version": "GOAE", "factor": 2.3, "quantity": 1, "service_date": "2026-08-29"}
    ]}
    """,
    "mutual_exclusion": """
    {"patient_id": "p-103", "line_items": [
        {"code": "1", "catalogue_version": "GOAE", "factor": 2.3, "quantity": 1, "service_date": "2026-08-29"},
        {"code": "5", "catalogue_version": "GOAE", "factor": 1.8, "quantity": 1, "service_date": "2026-08-29"}
    ]}
    """,
    "factor_out_of_range": """
    {"patient_id": "p-104", "line_items": [
        {"code": "1", "catalogue_version": "GOAE", "factor": 5.0, "quantity": 1, "service_date": "2026-08-29"}
    ]}
    """,
    "invalid_date_and_quantity": """
    {"patient_id": "p-105", "line_items": [
        {"code": "1", "catalogue_version": "GOAE", "factor": 2.3, "quantity": 5, "service_date": "2026-09-15"}
    ]}
    """,
}


def mock_ai_billing_agent(scenario: str) -> str:
    """Stand-in for a schema-constrained Gemini Pro call. Returns raw JSON
    text; calling this performs no side effect and bills nothing."""
    return MOCK_AGENT_RESPONSES[scenario]

## 3. Structured Output Parser — reject on parse failure

In [ ]:
class ParseResult(BaseModel):
    model_config = {"arbitrary_types_allowed": True}
    ok: bool
    claim: BillingClaim | None = None
    error: str | None = None


def parse_structured_output(raw_json: str) -> ParseResult:
    try:
        claim = BillingClaim.model_validate_json(raw_json)
    except ValidationError as exc:
        return ParseResult(ok=False, error=f"schema validation failed: {exc.error_count()} error(s)")
    except ValueError as exc:
        # model_validate_json raises ValueError (wrapping JSONDecodeError) on invalid JSON
        return ParseResult(ok=False, error=f"invalid JSON: {exc}")
    return ParseResult(ok=True, claim=claim)

## 4. Deterministic Catalogue Rules Engine

Nothing here depends on the model. All four checks are plain lookups and comparisons against a fixed catalogue.

In [ ]:
# Mock GOAE/GOZ catalogue: (code, catalogue_version) -> rules for that code
CATALOGUE = {
    ("1", "GOAE"): {"legal_factor_range": (1.0, 3.5), "max_quantity_per_day": 1},
    ("5", "GOAE"): {"legal_factor_range": (1.0, 2.5), "max_quantity_per_day": 1},
    ("12", "GOZ"): {"legal_factor_range": (1.0, 3.5), "max_quantity_per_day": 2},
}

# Codes that cannot be billed together on the same claim (symmetric)
MUTUAL_EXCLUSIONS = {
    "1": {"5"},
    "5": {"1"},
}

MAX_CLAIM_AGE_DAYS = 14


def rule_code_exists(item: BillingLineItem) -> str | None:
    if (item.code, item.catalogue_version) not in CATALOGUE:
        return f"code {item.code} does not exist in {item.catalogue_version} catalogue"
    return None


def rule_mutual_exclusions(claim: BillingClaim) -> str | None:
    codes = [item.code for item in claim.line_items]
    for code in codes:
        excluded = MUTUAL_EXCLUSIONS.get(code, set())
        conflicts = excluded & set(codes)
        if conflicts:
            return f"code {code} cannot be billed with {sorted(conflicts)} on the same claim"
    return None


def rule_factor_in_range(item: BillingLineItem) -> str | None:
    entry = CATALOGUE.get((item.code, item.catalogue_version))
    if entry is None:
        return None  # rule_code_exists already reports this
    low, high = entry["legal_factor_range"]
    if not (low <= item.factor <= high):
        return f"factor {item.factor} for code {item.code} outside legal range [{low}, {high}]"
    return None


def rule_date_and_quantity_valid(item: BillingLineItem, today: date) -> list[str]:
    issues = []
    if item.service_date > today:
        issues.append(f"service_date {item.service_date} is in the future")
    elif (today - item.service_date).days > MAX_CLAIM_AGE_DAYS:
        issues.append(f"service_date {item.service_date} is older than {MAX_CLAIM_AGE_DAYS} days")

    entry = CATALOGUE.get((item.code, item.catalogue_version))
    if entry is not None and item.quantity > entry["max_quantity_per_day"]:
        issues.append(f"quantity {item.quantity} exceeds max {entry['max_quantity_per_day']} per day for code {item.code}")
    return issues


def evaluate_claim(claim: BillingClaim, today: date = TODAY) -> list[str]:
    violations = []
    exclusion_violation = rule_mutual_exclusions(claim)
    if exclusion_violation:
        violations.append(exclusion_violation)
    for item in claim.line_items:
        for rule in (rule_code_exists, rule_factor_in_range):
            violation = rule(item)
            if violation:
                violations.append(violation)
        violations.extend(rule_date_and_quantity_valid(item, today))
    return violations

## 5. Full pipeline: agent → parser → rules engine

In [ ]:
def process_ai_billing_output(scenario: str) -> None:
    raw_json = mock_ai_billing_agent(scenario)
    parsed = parse_structured_output(raw_json)

    print(f"--- scenario: {scenario} ---")
    if not parsed.ok:
        print(f"REJECTED at parser: {parsed.error}\n")
        return

    violations = evaluate_claim(parsed.claim)
    if violations:
        print("REJECTED by rules engine:")
        for v in violations:
            print(f"  - {v}")
        print()
    else:
        print(f"APPROVED for billing: {parsed.claim.model_dump()}\n")


for scenario in MOCK_AGENT_RESPONSES:
    process_ai_billing_output(scenario)